In [ ]:
import sys
import os
import importlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

In [ ]:
# reload import when need
import src.eda.eda as eda
importlib.reload(eda)

In [ ]:
sys.path.append(os.path.abspath(".."))

from src.data.data_loader import load_data, prepare_and_save
from src.eda.eda import (
    get_all_caption,
    plot_caption_length,
    plot_top_words,
    show_sample,
    get_vocab,
    plot_unique_words_per_caption,
    plot_word_frequency,
    plot_ttr_distribution,
    find_samples_by_keyword
)

In [ ]:
# if run for 1st time then execute code below
prepare_and_save()

In [ ]:
# split data
train, val, test = load_data()
data = train

I. Phân tích dữ liệu

In [ ]:
# Total captions
captions = get_all_caption(data)
print("Total captions:", len(captions))

# sample data form dataset
show_sample(data)

# vocabulary size: total diff words
vocab = get_vocab(captions)
print("Vocabulary size:", len(vocab))

In [ ]:
# diagrams
plot_caption_length(captions, "outputs/figures/length.png")
img = mpimg.imread("outputs/figures/length.png")
plt.imshow(img)
plt.axis('off')
plt.show()
plot_top_words(captions, "outputs/figures/top_words.png")
img = mpimg.imread("outputs/figures/top_words.png")
plt.imshow(img)
plt.axis('off')
plt.show()

Giảỉ thích biểu đồ

In [ ]:
# unique words per caption
plot_unique_words_per_caption(captions,"outputs/figures/unique_words.png")
img = mpimg.imread("outputs/figures/unique_words.png")
plt.imshow(img)
plt.axis('off')
plt.show()

Biểu đồ tần suất xuất hiện của từ

In [ ]:
# unique words per caption
plot_word_frequency(captions,"outputs/figures/word_frequency.png")
img = mpimg.imread("outputs/figures/word_frequency.png")
plt.imshow(img)
plt.axis('off')
plt.show()

Biểu đồ độ đa dạng của từ


In [ ]:
plot_ttr_distribution(captions,"outputs/figures/ttr_distribution.png")
img = mpimg.imread("outputs/figures/ttr_distribution.png")
plt.imshow(img)
plt.axis('off')
plt.show()

Giải thích biểu đồ

TTR=Number of unique words​/Total number of words

là chỉ số đo độ đa dạng của từ vựng

Trong tập dữ liệu hiện tại thì TTR đang phân bố từ 0.7~1.0, đây là giá trị cao. Điều này cho thấy lượng từ vựng trong dataset rất đa dạng

Minh họa sự đa dạng hình ảnh trong tập dataset

In [ ]:
samples = []
samples += find_samples_by_keyword(train, "dog")
samples += find_samples_by_keyword(train, "car")
samples += find_samples_by_keyword(train, "beach")
samples += find_samples_by_keyword(train, "people")

for item in samples:
    plt.imshow(item["image"])
    plt.title(item["caption"])
    plt.axis("off")
    plt.show()

II. Tiền xử lý dữ liệu  
1. Làm sạch dữ liệu

In [ ]:
# reload import when need
import src.data.preprocessing as preprocessing
importlib.reload(preprocessing)

In [ ]:
from src.data.preprocessing import (
    clean_data,
    show_cleaning_exsample,
    remove_invalid
    
)

In [ ]:
original_train = train
cleaned_train=clean_data(train)

show_cleaning_exsample(original_train, 0)

2. Xử lý missing value  
Tiến hành kiểm tra và loại bỏ các caption rỗng hoặc không hợp lệ nhằm đảm bảo chất lượng dữ liệu.

In [ ]:
print("Before:", len(cleaned_train))
remove_invalid_train=remove_invalid(cleaned_train)
print("After:", len(remove_invalid_train))

Dựa vào kết quả trên thì ta có thể thấy dataset không chứa caption rỗng, tức là mỗi ảnh đều có ít nhất 1 caption

3. Feature Engineering

In [ ]:
from src.data.feature_engineering import *

captions = get_all_captions(cleaned_train)

vocab = build_vocab(captions)
word2idx = build_word2idx(vocab)

encoded_train = encode_dataset(cleaned_train, word2idx)

Sample demo

In [ ]:
sample = cleaned_train[0]["caption"][0]

print("Caption:", sample)
print("Encoded:", encode_caption(sample, word2idx))

4. Chuẩn hóa dữ liệu  
Ảnh được resize về kích thước cố định và chuẩn hóa giá trị pixel về khoảng [0, 1].
Điều này giúp mô hình học ổn định và nhanh hội tụ hơn.

In [ ]:
from src.data.normalize import *

sample = train[0]["image"]

print("Before:", np.array(sample).shape)

processed = process_image(sample)

print("After:", processed.shape)
print("Min:", processed.min(), "Max:", processed.max())

Pipeline hoàn chỉnh khi thực hiện tiền xử lý data

In [ ]:
from src.data.preprocessing import prepare_data
from src.data.feature_engineering import *
from src.data.dataset import FlickrDataset
from torch.utils.data import DataLoader

captions = get_all_captions(train)
vocab = build_vocab(captions)
word2idx = build_word2idx(vocab)

processed_train = prepare_data(train, word2idx)

dataset = FlickrDataset(processed_train)

loader = DataLoader(dataset, batch_size=32, shuffle=True)